# Clase 106 — Ray Tune para HPO distribuido

Ray puede ser pesado de instalar en Windows. Usamos `try/except` y caemos a Optuna como fallback (mismo concepto).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Función objetivo de juguete: paraboloide 2D con mínimo desplazado en (2, -1)
def objective(config):
    x, y = config['x'], config['y']
    # noise simula stochasticidad de entrenamiento
    noise = np.random.normal(0, 0.05)
    return (x - 2)**2 + (y + 1)**2 + noise

print('óptimo conocido: (x=2, y=-1) → loss=0')

## 1. Ray Tune (preferido) o Optuna (fallback)

In [ ]:
USE_RAY = False
try:
    from ray import tune
    from ray.tune.search.hyperopt import HyperOptSearch
    USE_RAY = True
    print('Ray disponible:', tune.__name__)
except Exception as e:
    print('Ray no disponible; usando Optuna como fallback. Motivo:', type(e).__name__)
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2. Search space + búsqueda aleatoria (baseline)

In [ ]:
rng = np.random.default_rng(42)

# uniform(-5, 5) x uniform(-5, 5); choice de optimizer (categórico ilustrativo)
n_trials = 40
random_trials = []
for t in range(n_trials):
    cfg = {'x': rng.uniform(-5, 5), 'y': rng.uniform(-5, 5), 'opt': rng.choice(['sgd', 'adam'])}
    loss = objective(cfg)
    random_trials.append((cfg, loss))

best_rand = min(random_trials, key=lambda t: t[1])
print(f'random search best: cfg={best_rand[0]}, loss={best_rand[1]:.4f}')

## 3. HyperOpt (TPE bayesiano) vía Optuna fallback

In [ ]:
if not USE_RAY:
    def optuna_obj(trial):
        x = trial.suggest_float('x', -5, 5)
        y = trial.suggest_float('y', -5, 5)
        opt = trial.suggest_categorical('opt', ['sgd', 'adam'])
        return objective({'x': x, 'y': y, 'opt': opt})
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(optuna_obj, n_trials=n_trials, show_progress_bar=False)
    print(f'TPE best: params={study.best_params}, loss={study.best_value:.4f}')
    bayes_trials = [(t.params, t.value) for t in study.trials]
else:
    search_space = {'x': tune.uniform(-5, 5), 'y': tune.uniform(-5, 5), 'opt': tune.choice(['sgd', 'adam'])}
    analysis = tune.run(lambda c: tune.report(loss=objective(c)), config=search_space, num_samples=n_trials, search_alg=HyperOptSearch(metric='loss', mode='min'), verbose=0)
    print('best:', analysis.get_best_config('loss', 'min'))
    bayes_trials = [(t.config, t.last_result['loss']) for t in analysis.trials]

## 4. ASHA scheduler (conceptual)

ASHA = Async Successive Halving. Idea: cada trial corre `r` epochs; cada `eta` (=3) trials, mantenemos el top-1/eta y duplicamos su budget. Esto descarta hiperparámetros malos rápido.

```python
from ray.tune.schedulers import ASHAScheduler
scheduler = ASHAScheduler(metric='loss', mode='min', max_t=100, grace_period=10, reduction_factor=3)
```

Simulamos el efecto: trials malos en epoch=10 se cortan.

In [ ]:
# ASHA simulado: cortamos trials cuyo loss en epoch=10 está sobre el 33% percentil
n_sim = 30
epoch_10_losses = rng.uniform(0.5, 50, n_sim)
epoch_100_losses = epoch_10_losses + rng.normal(0, 5, n_sim)

thresh = np.percentile(epoch_10_losses, 33)
survivors = epoch_10_losses <= thresh
print(f'ASHA: corre {n_sim} trials → mata {(~survivors).sum()} en epoch=10')
print(f'budget ahorrado: {(~survivors).sum() * 90} epochs (vs full grid)')

## 5. Visualización de la búsqueda

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
rx = [t[0]['x'] for t in random_trials]; ry = [t[0]['y'] for t in random_trials]
rl = [t[1] for t in random_trials]
bx = [t[0]['x'] for t in bayes_trials]; by_ = [t[0]['y'] for t in bayes_trials]
bl = [t[1] for t in bayes_trials]

axes[0].scatter(rx, ry, c=rl, cmap='viridis', s=40)
axes[0].scatter([2], [-1], marker='*', s=200, c='red', label='óptimo')
axes[0].set_title('Random search'); axes[0].legend()
axes[1].scatter(bx, by_, c=bl, cmap='viridis', s=40)
axes[1].scatter([2], [-1], marker='*', s=200, c='red')
axes[1].set_title('TPE/HyperOpt (bayesiano)')
plt.tight_layout(); plt.show()

## Conclusiones

- Ray Tune permite paralelizar HPO a múltiples nodos; en single-machine, Optuna alcanza.
- TPE / BayesOpt convergen más rápido que random search.
- ASHA + early stopping ahorran 50-80% del budget en deep learning real.
- Ver la Clase 105 (Keras Tuner) para integrar HPO directo con `model.fit()`.